In [ ]:
# Install required packages
!pip install --upgrade pip
!pip install haystack-ai==2.2.0 json-repair pytesseract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [haystack-ai]


# Gemini API Configuration

In [ ]:
from google import genai
from google.colab import userdata

gemini_api_key = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=gemini_api_key)
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Explain how AI works in a few words and funny way"
)
print(response.text)


AI is like teaching a toaster to make toast, but instead of programming it directly, you just show it a million pictures of toast and hope it figures it out. Sometimes it makes perfect golden-brown slices, and sometimes... well, you get charcoal.



# Loading Images

In [ ]:
# Upload image
from google.colab import files
uploaded = files.upload()

In [ ]:
img_path = list(uploaded.keys())
img_path

['Bees_fun_facts.jpg', 'ML_notes_example.png']

In [ ]:
# Test pytesseract
from PIL import Image
import pytesseract
img = Image.open(img_path[1])
extracted_text = pytesseract.image_to_string(img)
print(extracted_text)

 

 

Machine Learning

MACHINE LEARNING

Introduction

Introduction: Ever since computers were invented, we have wondered whether they might be
made to leam.If we could understand how to program them to leam-to improve automatically

with experience-the impact would be dramatic.

Imagine computers leamming from medical records which treatments are most effective
for new diseases

Houses leaming from experience to optimize energy costs based on the particular usage
pattems of their occupants.

Personal software assistants leaming the evolving interests of their users in order to
highlight especially relevant stories from the online moming newspaper

A successful understanding of how to make computers leam would open up many new uses
of computers and new levels of competence and customization

Some successful applications of machine learning

Leaming to recognize spoken words

Leaming to drive an autonomous vehicle
Leaming to classify new astronomical structures
Leaming to play world-cl

# Data Preprocessing
- Remove repeated whitespace (tabs/newlines)
- Trim leading/trailing spaces
- Preserve casing (crucial for proper nouns/context)
- Keep punctuation (maintains sentence structure)

In [ ]:
# Testing Preprocessing
import re
text = extracted_text
text = re.sub(r'\s+', ' ', text).strip()
print(text)

Machine Learning MACHINE LEARNING Introduction Introduction: Ever since computers were invented, we have wondered whether they might be made to leam.If we could understand how to program them to leam-to improve automatically with experience-the impact would be dramatic. Imagine computers leamming from medical records which treatments are most effective for new diseases Houses leaming from experience to optimize energy costs based on the particular usage pattems of their occupants. Personal software assistants leaming the evolving interests of their users in order to highlight especially relevant stories from the online moming newspaper A successful understanding of how to make computers leam would open up many new uses of computers and new levels of competence and customization Some successful applications of machine learning Leaming to recognize spoken words Leaming to drive an autonomous vehicle Leaming to classify new astronomical structures Leaming to play world-class backgammon Wh

# LLM Questions Generation Pipeline

In [ ]:
# import all required packages
import json, json_repair, os,  random
from pprint import pprint
from getpass import getpass
from typing import List, Dict
from haystack.components.builders import PromptBuilder
from haystack.utils import Secret
from haystack import Pipeline, Document, component
# from PIL import Image
# import pytesseract
# import re

In [ ]:
# Create a component to extract text from images
@component
class ImageToTextExtractor:
  @component.output_types(documents=list[Document])
  def run(self, file_path: str):
    img = Image.open(file_path)
    text = pytesseract.image_to_string(img)
    doc = Document(content=text, meta={"source": file_path})
    return {"documents": [doc]}

# Create a component to perform minimal preprecessing on extracted text
@component
class TextPreprocessor:
    @component.output_types(documents=list[Document])
    def run(self, documents: list[Document]):
        processed_docs = []
        for doc in documents:
            text = doc.content
            text = re.sub(r'\s+', ' ', text).strip()
            processed_docs.append(Document(content=text, meta=doc.meta))
        return {"documents": processed_docs}

# Create a component for Gemini model instead of using Haystack's default model (openAI)
@component
class GeminiGenerator:
  def __init__(self, api_key: str, model: str):
    self.api_key = api_key
    self.model = model

  @component.output_types(gemini_replies=list[str])
  def run(self, prompt: str):

    client = genai.Client(api_key=self.api_key)

    response = client.models.generate_content(
        model=self.model,
        contents=prompt
    )
    return {"gemini_replies": [response.text]}

# Create a component to parse irrelevant text (keep only json)
@component
class QuizParser:
    @component.output_types(quiz=dict)
    def run(self, replies: list[str]):
        reply = replies[0]
        first_index = min(reply.find("{"), reply.find("["))
        last_index = max(reply.rfind("}"), reply.rfind("]")) + 1
        json_portion = reply[first_index:last_index]

        try:
            quiz = json.loads(json_portion)
        except json.JSONDecodeError:
            quiz = json_repair.loads(json_portion)
        if isinstance(quiz, list):
            quiz = quiz[0]
        return {"quiz": quiz}

In [ ]:
os.environ["GEMINI_API_KEY"] = gemini_api_key

if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter GEMINI API key:")

In [ ]:
quiz_generation_template = """Given the following text, create 5 multiple choice quizzes in JSON format.
Each question should have 4 different options, and only one of them should be correct.
The options should be unambiguous.
Each option should begin with a letter followed by a period and a space (e.g., "a. option").
The question should also briefly mention the general topic of the text so that it can be understood in isolation.
Each question should not give hints to answer the other questions.
Include challenging questions, which require reasoning.

respond with JSON only, no markdown or descriptions.

example JSON format you should absolutely follow:
{"topic": "a sentence explaining the topic of the text",
 "questions":
  [
    {
      "question": "text of the question",
      "options": ["a. 1st option", "b. 2nd option", "c. 3rd option", "d. 4th option"],
      "right_option": "c"  # letter of the right option ("a" for the first, "b" for the second, etc.)
    }, ...
  ]
}

text:
{% for doc in documents %}{{ doc.content }}{% endfor %}
"""

quiz_generation_pipeline = Pipeline()
quiz_generation_pipeline.add_component("text_extractor", ImageToTextExtractor())
quiz_generation_pipeline.add_component("text_preprocessor", TextPreprocessor())
quiz_generation_pipeline.add_component("prompt_builder", PromptBuilder(template=quiz_generation_template))
quiz_generation_pipeline.add_component("gemini_generator", GeminiGenerator(api_key=os.environ["GEMINI_API_KEY"], model="gemini-2.0-flash"))
quiz_generation_pipeline.add_component("parser", QuizParser())

quiz_generation_pipeline.connect("text_extractor", "text_preprocessor")
quiz_generation_pipeline.connect("text_preprocessor", "prompt_builder")
quiz_generation_pipeline.connect("prompt_builder", "gemini_generator")
quiz_generation_pipeline.connect("gemini_generator", "parser")

🚅 Components
  - text_extractor: ImageToTextExtractor
  - text_preprocessor: TextPreprocessor
  - prompt_builder: PromptBuilder
  - gemini_generator: GeminiGenerator
  - parser: QuizParser
🛤️ Connections
  - text_extractor.documents -> text_preprocessor.documents (list[Document])
  - text_preprocessor.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> gemini_generator.prompt (str)
  - gemini_generator.gemini_replies -> parser.replies (list[str])

In [ ]:
quiz_generation_pipeline.show()

NameError: name 'quiz_generation_pipeline' is not defined

In [ ]:
quiz = quiz_generation_pipeline.run({"text_extractor": {"file_path": img_path[1]}})

In [ ]:
# print(quiz["prompt_builder"])
# pprint(quiz["gemini_generator"]["gemini_replies"])

NameError: name 'pprint' is not defined

In [ ]:
pprint(quiz['parser']['quiz'])

{'questions': [{'options': ['a. Eliminating the need for computer programmers.',
                            'b. Automatically improving with experience.',
                            'c. Reducing the amount of data required for '
                            'analysis.',
                            'd. Replacing traditional statistical methods.'],
                'question': 'According to the text, what is a key benefit of '
                            'machine learning?',
                'right_option': 'b'},
               {'options': ['a. Learning to classify new astronomical '
                            'structures.',
                            'b. Learning to recognize spoken words.',
                            'c. Learning to drive an autonomous vehicle.',
                            'd. Learning to predict stock market '
                            'fluctuations.'],
                'question': 'The text mentions applications of machine '
                            'learning.

# Evaluate Student answers

In [ ]:
!pip install -q -U google-generativeai



In [ ]:
!pip install -q -U google-generativeaigoogle-generatgoogle-generativeaigoogle-



In [ ]:
# Import necessary libraries
import google.generativeai as genai
import json
import os
from IPython.display import Markdown
from google.colab import userdata

# 1. Configure Gemini API (This part should be handled by Maram, but included for completeness)
# Ensure your GEMINI_API_KEY is stored in Colab Secrets
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
        genai.configure(api_key=GEMINI_API_KEY)
            print("Gemini API configured successfully!")
            except Exception as e:
                print(f"Error configuring Gemini API: {e}")
                    print("Please ensure 'GEMINI_API_KEY' is set in Colab Secrets.")

                    # Initialize the Gemini model for evaluation
                    # You might want to use a more powerful model like 'gemini-pro' for evaluation if available/necessary.
                    evaluation_model = genai.GenerativeModel('gemini-pro')

                    # --- Function to Evaluate Student Answers ---
                    def evaluate_student_answer(question: str, correct_answer: str, student_answer: str) -> dict:
                        """
                            Evaluates a student's answer against the correct answer using an LLM.

                                Args:
                                        question (str): The question asked.
                                                correct_answer (str): The expected correct answer.
                                                        student_answer (str): The answer provided by the student.

                                                            Returns:
                                                                    dict: A dictionary containing evaluation feedback, score, and a pass/fail status.
                                                                        """

                                                                            prompt = f"""
                                                                                You are an AI assistant designed to evaluate student answers for a quiz.
                                                                                    Your task is to compare a student's answer to a correct answer for a given question.
                                                                                        Provide constructive feedback and assign a score out of 5, where 5 is a perfect answer and 0 is completely incorrect.
                                                                                            Also, determine if the student passed this specific question (score >= 3 is a pass).

                                                                                                Here is the information:
                                                                                                    Question: {question}
                                                                                                        Correct Answer: {correct_answer}
                                                                                                            Student's Answer: {student_answer}

                                                                                                                Your output should be a JSON object with the following keys:
                                                                                                                    - "feedback": A string explaining why the answer was correct, partially correct, or incorrect.
                                                                                                                        - "score": An integer from 0 to 5.
                                                                                                                            - "passed": A boolean (true if score >= 3, false otherwise).
                                                                                                                                - "reasoning": (Optional) A brief explanation for the score given.

                                                                                                                                    Example JSON output:
                                                                                                                                        {{
                                                                                                                                              "feedback": "Your answer was accurate and covered the key points.",
                                                                                                                                                    "score": 5,
                                                                                                                                                          "passed": true,
                                                                                                                                                                "reasoning": "The student correctly identified all main components."
                                                                                                                                                                    }}
                                                                                                                                                                        """
                                                                                                                                                                            try:
                                                                                                                                                                                    response = evaluation_model.generate_content(prompt)
                                                                                                                                                                                            # Assuming the LLM is good at returning JSON, we try to parse it.
                                                                                                                                                                                                    # It's good practice to add robust error handling for JSON parsing.
                                                                                                                                                                                                            evaluation_result = json.loads(response.text)
                                                                                                                                                                                                                    return evaluation_result
                                                                                                                                                                                                                        except json.JSONDecodeError as e:
                                                                                                                                                                                                                                print(f"Error decoding JSON from LLM response: {e}")
                                                                                                                                                                                                                                        print(f"LLM raw response: {response.text}")
                                                                                                                                                                                                                                                return {
                                                                                                                                                                                                                                                            "feedback": "Could not parse evaluation. Please check LLM response.",
                                                                                                                                                                                                                                                                        "score": 0,
                                                                                                                                                                                                                                                                                    "passed": False,
                                                                                                                                                                                                                                                                                                "reasoning": f"JSON decoding error: {e}"
                                                                                                                                                                                                                                                                                                        }
                                                                                                                                                                                                                                                                                                            except Exception as e:
                                                                                                                                                                                                                                                                                                                    print(f"An error occurred during evaluation: {e}")
                                                                                                                                                                                                                                                                                                                            return {
                                                                                                                                                                                                                                                                                                                                        "feedback": f"An unexpected error occurred: {e}",
                                                                                                                                                                                                                                                                                                                                                    "score": 0,
                                                                                                                                                                                                                                                                                                                                                                "passed": False,
                                                                                                                                                                                                                                                                                                                                                                            "reasoning": "Internal error during evaluation."
                                                                                                                                                                                                                                                                                                                                                                                    }

                                                                                                                                                                                                                                                                                                                                                                                    # --- Example Usage ---
                                                                                                                                                                                                                                                                                                                                                                                    if __name__ == "__main__":
                                                                                                                                                                                                                                                                                                                                                                                        # This is placeholder data. In a real scenario,
                                                                                                                                                                                                                                                                                                                                                                                            # these would come from the quiz generation (Tarig's part) and student input.

                                                                                                                                                                                                                                                                                                                                                                                                # Example 1: Correct Answer
                                                                                                                                                                                                                                                                                                                                                                                                    q1 = "What is the capital of France?"
                                                                                                                                                                                                                                                                                                                                                                                                        correct_a1 = "Paris"
                                                                                                                                                                                                                                                                                                                                                                                                            student_a1 = "Paris"
                                                                                                                                                                                                                                                                                                                                                                                                                evaluation1 = evaluate_student_answer(q1, correct_a1, student_a1)
                                                                                                                                                                                                                                                                                                                                                                                                                    print("\n--- Evaluation 1 ---")
                                                                                                                                                                                                                                                                                                                                                                                                                        print(json.dumps(evaluation1, indent=2, ensure_ascii=False))

                                                                                                                                                                                                                                                                                                                                                                                                                            # Example 2: Partially Correct/Similar Answer
                                                                                                                                                                                                                                                                                                                                                                                                                                q2 = "Explain the concept of Artificial Intelligence in brief."
                                                                                                                                                                                                                                                                                                                                                                                                                                    correct_a2 = "Artificial Intelligence (AI) is a field of computer science that aims to create machines that can perform tasks that typically require human intelligence."
                                                                                                                                                                                                                                                                                                                                                                                                                                        student_a2 = "AI is when computers act like humans."
                                                                                                                                                                                                                                                                                                                                                                                                                                            evaluation2 = evaluate_student_answer(q2, correct_a2, student_a2)
                                                                                                                                                                                                                                                                                                                                                                                                                                                print("\n--- Evaluation 2 ---")
                                                                                                                                                                                                                                                                                                                                                                                                                                                    print(json.dumps(evaluation2, indent=2, ensure_ascii=False))

                                                                                                                                                                                                                                                                                                                                                                                                                                                        # Example 3: Incorrect Answer
                                                                                                                                                                                                                                                                                                                                                                                                                                                            q3 = "Which planet is known as the Red Planet?"
                                                                                                                                                                                                                                                                                                                                                                                                                                                                correct_a3 = "Mars"
                                                                                                                                                                                                                                                                                                                                                                                                                                                                    student_a3 = "Jupiter"
                                                                                                                                                                                                                                                                                                                                                                                                                                                                        evaluation3 = evaluate_student_answer(q3, correct_a3, student_a3)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                            print("\n--- Evaluation 3 ---")
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                print(json.dumps(evaluation3, indent=2, ensure_ascii=False))

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    # Example 4: Multiple Choice Scenario (assuming correct_answer holds the correct option letter)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        # This would require more sophisticated parsing of question and options if LLM generates MCQs.
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            # For now, we assume correct_answer is the text of the correct choice.
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                q4 = "What is the primary function of a CPU?"
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    correct_a4 = "To execute instructions and perform calculations."
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        student_a4 = "It processes data."
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            evaluation4 = evaluate_student_answer(q4, correct_a4, student_a4)
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                print("\n--- Evaluation 4 ---")
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    print(json.dumps(evaluation4, indent=2, ensure_ascii=False))


How to run ?


1.  Pick the first quiz question from a JSON object.
2.  Display the question and options to the student (clean style).
3.  User Input() + Use evaluate_student_answer() to send the question, correct answer, and student’s answer for evaluation.
4.  Print the evaluation result in a clean, readable JSON format.
